In [ ]:
import torch
import os
import logging
from torchvision import transforms


from src.dataset.Dataset import VisDrone
from src.dataset.collate import collate_fn

from src.cnn.VisDroneCNN import VisDroneCNN

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logger = logging.getLogger(__name__)

In [ ]:
print(os.getcwd())
train_dev_data = "data/VisDrone_Dataset/VisDrone2019-DET-val/images"
train_labels = "data/VisDrone_Dataset/VisDrone2019-DET-val/labels"

val_data = "data/VisDrone_Dataset/VisDrone2019-DET-val/images"
val_labels = "data/VisDrone_Dataset/VisDrone2019-DET-val/labels"


In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((32, 32))
])
dtype = torch.float32

In [ ]:
visdrone = VisDrone(data_dir=train_dev_data, labels_dir=train_labels, transform=transform, dtype=dtype)
dataset = torch.utils.data.DataLoader(visdrone,
                                      batch_size=32,
                                      shuffle=True,
                                      num_workers=4,
                                      collate_fn=collate_fn,
                                      pin_memory=True,
                                      prefetch_factor=2)

visdrone_val = VisDrone(data_dir=val_data, labels_dir=val_labels, transform=transform, dtype=dtype)
dataset_val = torch.utils.data.DataLoader(visdrone_val,
                                          batch_size=32,
                                          shuffle=True,
                                          num_workers=4,
                                          collate_fn=collate_fn,
                                          pin_memory=True,
                                          prefetch_factor=2)

In [ ]:
print(torch.cuda.get_device_name(0))
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
cnn = VisDroneCNN(B_boxes=1, C=10, dtype=dtype)
cnn = cnn.to(device)
opt = torch.optim.Adam(cnn.parameters())

In [ ]:
wandb_config = {"TEST": "TEST"}
cnn.fit(epochs=5,
        optimizer=opt,
        train_loader=dataset,
        val_loader=dataset_val,
        wandb_config=wandb_config)
#torch.save(cnn.state_dict(), "first.pth")